# Notebook 02 (Participant): Evaluation + Metrics

Complete the `TODO` sections to evaluate generated designs and export metrics.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'pandas', 'matplotlib', 'wandb']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from engibench.problems.beams2d.v0 import Beams2D

# Optional W&B artifact flow (disabled by default)
USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'


def preferred_artifact_dir() -> Path:
    if 'google.colab' in sys.modules:
        return Path('/content/dcc26_artifacts')
    return Path('workshops/dcc26/artifacts')


ARTIFACT_DIR = preferred_artifact_dir()
required = [
    ARTIFACT_DIR / 'generated_designs.npy',
    ARTIFACT_DIR / 'baseline_designs.npy',
    ARTIFACT_DIR / 'conditions.json',
]

if not all(p.exists() for p in required):
    if USE_WANDB_ARTIFACTS:
        try:
            import wandb

            ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
            run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, job_type='artifact-download', reinit=True)
            if WANDB_ENTITY:
                artifact_ref = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_ARTIFACT_NAME}:{WANDB_ARTIFACT_ALIAS}"
            else:
                artifact_ref = f"{WANDB_PROJECT}/{WANDB_ARTIFACT_NAME}:{WANDB_ARTIFACT_ALIAS}"
            artifact = run.use_artifact(artifact_ref, type='dataset')
            artifact.download(root=str(ARTIFACT_DIR))
            run.finish()
            print('Downloaded artifacts from W&B to', ARTIFACT_DIR)
        except Exception as exc:
            raise FileNotFoundError(
                'Artifacts missing locally and W&B download failed. '
                'Run Notebook 01 first or disable USE_WANDB_ARTIFACTS. '
                f'Details: {exc}'
            ) from exc
    else:
        missing = '\n'.join(f'- {p}' for p in required if not p.exists())
        raise FileNotFoundError(
            'Notebook 01 artifacts not found. Run Notebook 01 first (including export cell). '
            'If you want remote restore, enable USE_WANDB_ARTIFACTS. Missing files:\n' + missing
        )

print('using artifact dir:', ARTIFACT_DIR)

gen_designs = np.load(ARTIFACT_DIR / 'generated_designs.npy')
baseline_designs = np.load(ARTIFACT_DIR / 'baseline_designs.npy')
with open(ARTIFACT_DIR / 'conditions.json', encoding='utf-8') as f:
    conditions = json.load(f)

print('generated:', gen_designs.shape)
print('baseline:', baseline_designs.shape)
print('conditions:', len(conditions))


In [ ]:
problem = Beams2D(seed=7)

# TODO 1: implement per-sample evaluation loop
# For each (generated, baseline, config):
# - check constraints for generated and baseline
# - run problem.simulate for each
# - append dict rows to `rows` with:
#   sample, gen_obj, base_obj, gen_minus_base, gen_violations, base_violations

rows = []
raise NotImplementedError('Complete TODO 1 evaluation loop')

results = pd.DataFrame(rows)
results.head()

In [ ]:
# TODO 2: implement summary metrics
# - n_samples
# - gen_obj_mean
# - base_obj_mean
# - improvement_rate
# - gen_violation_ratio
# - base_violation_ratio
# - gen_diversity_l2 (use helper below)

def mean_pairwise_l2(designs: np.ndarray) -> float:
    flat = designs.reshape(designs.shape[0], -1)
    n = flat.shape[0]
    if n < 2:
        return 0.0
    dists = []
    for i in range(n):
        for j in range(i + 1, n):
            dists.append(float(np.linalg.norm(flat[i] - flat[j])))
    return float(np.mean(dists))

raise NotImplementedError('Complete TODO 2 summary metrics')

In [ ]:
results.to_csv(ARTIFACT_DIR / 'per_sample_metrics.csv', index=False)
summary_df.to_csv(ARTIFACT_DIR / 'metrics_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(results['gen_obj'], bins=10, alpha=0.7, label='generated')
ax.hist(results['base_obj'], bins=10, alpha=0.7, label='baseline')
ax.set_xlabel('Compliance objective (lower is better)')
ax.set_ylabel('Count')
ax.set_title('Generated vs baseline objective distribution')
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'objective_histogram.png', dpi=150)
plt.show()

print('Saved:')
print('-', ARTIFACT_DIR / 'per_sample_metrics.csv')
print('-', ARTIFACT_DIR / 'metrics_summary.csv')
print('-', ARTIFACT_DIR / 'objective_histogram.png')

In [ ]:
# Visual side-by-side sample grid
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
for i, ax in enumerate(axes.ravel()):
    if i >= 12:
        break
    pair_idx = i // 2
    if i % 2 == 0:
        ax.imshow(gen_designs[pair_idx], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'gen {pair_idx}')
    else:
        ax.imshow(baseline_designs[pair_idx], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'base {pair_idx}')
    ax.axis('off')
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'design_grid.png', dpi=150)
plt.show()

## Interpretation hints

- `improvement_rate` shows how often generated designs beat baselines on objective value.
- `gen_violation_ratio` tracks practical feasibility pressure from constraints.
- `gen_diversity_l2` is a simple diversity proxy across generated designs.
